In [23]:
#imports 

import numpy as np
import pandas as pd

import matplotlib.pyplot as plt

In [24]:
# get the data

import os

if not os.path.exists('course_lead_scoring.csv'):
    !wget https://raw.githubusercontent.com/alexeygrigorev/datasets/master/course_lead_scoring.csv
else:
    print("File already exists — skipping download.")

File already exists — skipping download.


In [25]:
df = pd.read_csv('course_lead_scoring.csv')
df.head()

,lead_source,industry,number_of_courses_viewed,annual_income,employment_status,location,interaction_count,lead_score,converted
0,paid_ads,NaN,1,79450.0,unemployed,south_america,4,0.94,1
1,social_media,retail,1,46992.0,employed,south_america,1,0.80,0
2,events,healthcare,5,78796.0,unemployed,australia,3,0.69,1
3,paid_ads,retail,2,83843.0,NaN,australia,1,0.87,0
4,referral,education,3,85012.0,self_employed,europe,3,0.62,1


In [26]:
# data preparation - exploration

# transposes the dataframe to see more columns
df.head().T

,0,1,2,3,4
lead_source,paid_ads,social_media,events,paid_ads,referral
industry,NaN,retail,healthcare,retail,education
number_of_courses_viewed,1,1,5,2,3
annual_income,79450.0,46992.0,78796.0,83843.0,85012.0
employment_status,unemployed,employed,unemployed,NaN,self_employed
location,south_america,south_america,australia,australia,europe
interaction_count,4,1,3,1,3
lead_score,0.94,0.8,0.69,0.87,0.62
converted,1,0,1,0,1


In [27]:
df.columns = df.columns.str.lower().str.replace(' ', '_')   # cleans column names

categorical_columns = list(df.dtypes[df.dtypes == 'object'].index)  # lists categorical columns

for c in categorical_columns:  
    df[c] = df[c].str.lower().str.replace(' ', '_')   # cleans categorical values

df.head()

,lead_source,industry,number_of_courses_viewed,annual_income,employment_status,location,interaction_count,lead_score,converted
0,paid_ads,NaN,1,79450.0,unemployed,south_america,4,0.94,1
1,social_media,retail,1,46992.0,employed,south_america,1,0.80,0
2,events,healthcare,5,78796.0,unemployed,australia,3,0.69,1
3,paid_ads,retail,2,83843.0,NaN,australia,1,0.87,0
4,referral,education,3,85012.0,self_employed,europe,3,0.62,1


In [28]:
df.dtypes # checks data types of columns


lead_source                  object
industry                     object
number_of_courses_viewed      int64
annual_income               float64
employment_status            object
location                     object
interaction_count             int64
lead_score                  float64
converted                     int64
dtype: object

In [29]:
df.isnull().sum()  # checks for missing values

lead_source                 128
industry                    134
number_of_courses_viewed      0
annual_income               181
employment_status           100
location                     63
interaction_count             0
lead_score                    0
converted                     0
dtype: int64

In [30]:
df[categorical_columns] = df[categorical_columns].fillna('NA')  # fills missing values in categorical columns with 'NA'

df[categorical_columns].isnull().sum()  # checks again for missing values in categorical columns    

lead_source          0
industry             0
employment_status    0
location             0
dtype: int64

In [31]:
numerical_columns = df.select_dtypes(include=['number']).columns # gets list of numerical columns

df[numerical_columns] = df[numerical_columns].fillna(0.0)  # fills missing values in numerical columns with 0

df.isnull().sum()  # checks again for missing values in the dataframe 

lead_source                 0
industry                    0
number_of_courses_viewed    0
annual_income               0
employment_status           0
location                    0
interaction_count           0
lead_score                  0
converted                   0
dtype: int64

In [32]:
# Q1 - most freequent value in 'industry' column

df['industry'].mode()[0]    # returns the most frequent value in the 'industry' column

'retail'

In [33]:
# Q2 - correlation matrix

corr_matrix = df.corr(numeric_only=True)

corr_matrix

,number_of_courses_viewed,annual_income,interaction_count,lead_score,converted
number_of_courses_viewed,1.000000,0.009770,-0.023565,-0.004879,0.435914
annual_income,0.009770,1.000000,0.027036,0.015610,0.053131
interaction_count,-0.023565,0.027036,1.000000,0.009888,0.374573
lead_score,-0.004879,0.015610,0.009888,1.000000,0.193673
converted,0.435914,0.053131,0.374573,0.193673,1.000000


In [34]:
print("interaction_count vs lead_score:", corr_matrix.loc['interaction_count', 'lead_score'])
print("number_of_courses_viewed vs lead_score:", corr_matrix.loc['number_of_courses_viewed', 'lead_score'])
print("number_of_courses_viewed vs interaction_count:", corr_matrix.loc['number_of_courses_viewed', 'interaction_count'])
print("annual_income vs interaction_count:", corr_matrix.loc['annual_income', 'interaction_count'])

interaction_count vs lead_score: 0.009888182496913131
number_of_courses_viewed vs lead_score: -0.004878998354681276
number_of_courses_viewed vs interaction_count: -0.023565222882888037
annual_income vs interaction_count: 0.02703647240481443


In [35]:
# highest correlation is between annual_income and interaction_count (0.03)

In [36]:
# Split the data and add scikit-learn

from sklearn.model_selection import train_test_split

In [37]:
df_full_train, df_test = train_test_split(df, test_size=0.2, random_state=42) # splits the data into training and test sets

df_train, df_val = train_test_split(df_full_train, test_size=0.25, random_state=42) # splits the training data into training and validation sets - 0.25 x 0.8 = 0.2

len(df), len(df_train), len(df_val), len(df_test)  # checks the lengths of the datasets

(1462, 876, 293, 293)

In [38]:
df_train = df_train.reset_index(drop=True) 
df_val = df_val.reset_index(drop=True)
df_test = df_test.reset_index(drop=True)  # resets the indices of the datasets
df_full_train = df_full_train.reset_index(drop=True)  # resets the index of the full training dataset


In [39]:
df_train 

,lead_source,industry,number_of_courses_viewed,annual_income,employment_status,location,interaction_count,lead_score,converted
0,paid_ads,retail,0,58472.0,student,middle_east,5,0.03,0
1,organic_search,manufacturing,3,71738.0,student,middle_east,6,0.77,1
2,paid_ads,technology,3,81973.0,employed,north_america,2,0.59,1
3,NA,technology,1,74956.0,employed,europe,3,0.34,1
4,organic_search,retail,3,59335.0,student,australia,1,0.98,1
...,...,...,...,...,...,...,...,...,...
871,organic_search,other,1,43907.0,employed,australia,4,0.33,1
872,social_media,retail,3,64969.0,employed,north_america,1,0.18,0
873,NA,education,3,89042.0,employed,asia,4,0.75,1
874,social_media,manufacturing,1,0.0,self_employed,europe,1,0.65,0


In [40]:
# remove target value y from the dataframe

y_train = df_train['converted'].values
y_val = df_val['converted'].values
y_test = df_test['converted'].values

del df_train['converted']
del df_val['converted']
del df_test['converted']    

In [41]:
df_train

,lead_source,industry,number_of_courses_viewed,annual_income,employment_status,location,interaction_count,lead_score
0,paid_ads,retail,0,58472.0,student,middle_east,5,0.03
1,organic_search,manufacturing,3,71738.0,student,middle_east,6,0.77
2,paid_ads,technology,3,81973.0,employed,north_america,2,0.59
3,NA,technology,1,74956.0,employed,europe,3,0.34
4,organic_search,retail,3,59335.0,student,australia,1,0.98
...,...,...,...,...,...,...,...,...
871,organic_search,other,1,43907.0,employed,australia,4,0.33
872,social_media,retail,3,64969.0,employed,north_america,1,0.18
873,NA,education,3,89042.0,employed,asia,4,0.75
874,social_media,manufacturing,1,0.0,self_employed,europe,1,0.65


In [ ]:
df_full_train.isnull().sum()  # checks for missing values in the full training dataset

lead_source                 0
industry                    0
number_of_courses_viewed    0
annual_income               0
employment_status           0
location                    0
interaction_count           0
lead_score                  0
converted                   0
dtype: int64

In [ ]:
df_full_train.converted.value_counts() # checks the distribution of the target variable in the full training dataset

converted
1    710
0    459
Name: count, dtype: int64

In [45]:
df_full_train.converted.value_counts(normalize=True)  # checks the distribution of the target variable in the full training dataset

converted
1    0.607357
0    0.392643
Name: proportion, dtype: float64

In [50]:
global_converted_rate = df_full_train.converted.mean()  # calculates the mean of the target variable in the full training dataset
# notice mean is same as the proportion of 1s in the convererted 
round(global_converted_rate, 2)

np.float64(0.61)

In [51]:
# Q3 - calculate mutual infomation score

df_full_train.dtypes

lead_source                  object
industry                     object
number_of_courses_viewed      int64
annual_income               float64
employment_status            object
location                     object
interaction_count             int64
lead_score                  float64
converted                     int64
dtype: object

In [52]:
df_train.columns


Index(['lead_source', 'industry', 'number_of_courses_viewed', 'annual_income',
       'employment_status', 'location', 'interaction_count', 'lead_score'],
      dtype='object')

In [55]:
categorical = ['lead_source', 'industry', 'employment_status', 'location'] # defines the list of categorical columns

In [56]:
df_train[categorical].nunique()  # checks the number of unique values in each categorical column

lead_source          6
industry             8
employment_status    5
location             8
dtype: int64

In [57]:
from sklearn.metrics import mutual_info_score

In [58]:
for col in categorical:
    mi = mutual_info_score(df_train[col], y_train)
    print(f"{col}: {mi:.3f}")

lead_source: 0.035
industry: 0.012
employment_status: 0.013
location: 0.004


In [59]:
# Lead source has the highest mutual information score (0.035)